# 1- Business Problem

### Nowadays we have many recommendation algorithms everywhere, specially on streaming platforms, deciding for us what we are going to watch or hear next. The problem is often, either by the politics of the platforms or by the set up of those algorithms, we end up getting the same thing recommended over and over again, regardless of what you asked for.

### For our business problem, we are dealing with music streaming platforms, where many users claim to end up listening to the same random songs over and over again, one example is when they use the playlist generation feature from those platforms and the songs have nothing to do with what they asked for.

### To address this issue, the following algorithm was developed, aiming to provide music lovers with playlists where they can explore similar songs to the one they like, or ask for a playlist that simply match the vibe they are in.

# 2- Code

## 2.1 - Setup

### To run this project you will need the following libraries and list:
- tiktoken
- transformers
- torch
- langdetect
- nltk
- sentence_transformers
- SentencePiece

You can download all of those with: pip install "name of the libary"

In [1]:
# Importing the libraries needed
import os
import re
import numpy as np
import pandas as pd
import nltk

#To deal with the multiple language factor in our playlist
from langdetect import detect

#To help us normalize our words for analysis
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer, WordNetLemmatizer

from transformers import pipeline
from sentence_transformers import SentenceTransformer

#To help us build our matrix and calculate similarity
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

#To run fuzzy matching when creating our matrix
from rapidfuzz import process, fuzz

/Applications/anaconda3/envs/data_mining_class/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#If not yet downloaded, download the lists to help the nltk functions 

nltk.download("wordnet")
nltk.download("omw-1.4")


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/willianfranco/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/willianfranco/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 2.2- Data Loading

### For this step we use 2 different data sources:
- A folder with .txt files containing the lyrics from the songs (in this case, downloaded using the Genius API)
- A file containing the information about the songs in the folder, in this case, the JSON file from Apple Music containing informations such as: Artist, Song Name, Genre and others...

In [3]:
LYRICS_FOLDER = "lyrics"
apple_df = "Apple Music Library Tracks.json"

In [4]:
tracks_data = pd.read_json(apple_df)
tracks_data.head()

,Content Type,Track Identifier,Title,Sort Name,Artist,Sort Artist,Composer,Is Part of Compilation,Album,Sort Album,...,Remember Playback Position,Album Rating Method,Work Name,Movement Name,Movement Number,Movement Count,Display Work Name,Playlist Only Track,Favorite Status - Album,Favorite Date - Album
0,Song,182857262,Beg for You (feat. Rina Sawayama) [A. G. Cook ...,Beg for You (feat. Rina Sawayama) [A. G. Cook ...,Charli xcx,Charli xcx,"Alexander Soifer, Anoo. Bhagavan, BUMZU, Charl...",0.0,Beg For You (feat. Rina Sawayama) [The Remixes...,Beg For You (feat. Rina Sawayama) [The Remixes...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Song,182857266,Beg for You (feat. Rina Sawayama) [Jodie Harsh...,Beg for You (feat. Rina Sawayama) [Jodie Harsh...,Charli xcx,Charli xcx,"Alexander Soifer, Anoo. Bhagavan, Charlotte Ai...",0.0,Beg For You (feat. Rina Sawayama) [The Remixes...,Beg For You (feat. Rina Sawayama) [The Remixes...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Song,182857270,Beg for You (feat. Rina Sawayama) [Pocket Remix],Beg for You (feat. Rina Sawayama) [Pocket Remix],Charli xcx,Charli xcx,"Alexander Soifer, Anoo. Bhagavan, Charlotte Ai...",0.0,Beg For You (feat. Rina Sawayama) [The Remixes...,Beg For You (feat. Rina Sawayama) [The Remixes...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Song,182857474,Beg for You (feat. Rina Sawayama),Beg for You (feat. Rina Sawayama),Charli xcx,Charli xcx,"Alexander Soifer, Anoo. Bhagavan, Charlotte Ai...",0.0,Beg For You (feat. Rina Sawayama) [The Remixes...,Beg For You (feat. Rina Sawayama) [The Remixes...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Song,182857478,Acontece,Acontece,Jão,Jão,Jão & Pedro Tofani,0.0,PIRATA,PIRATA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### We have many columns from the apple dataset, but for our recommendation system we will use only the genre. The Title of the song and the Artist will also be used to match with the dataset containing the lyrics feature from the songs.

## 2.3- Models
### Sentiment und embedding model.

### Before starting the next stepts, we must initiate our embeding and sentiment models.

For Sentiment analysis we use "Roberta", the AI model trained to identify sentiment based on tweets.
For more information about the model: "https://medium.com/@monica.rotulo/tweets-sentiment-analysis-with-roberta-1f30cf4e1035"

As for embedding, since we have many languages and a massive volume of text in our playlist, we are using the "paraphrase-multilingual-MiniLM-L12-v2" sentence transformer model for multi lingual coverage and performance.

In [5]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

#Used to turn text into numeric vector, planned to help with the semantic search.
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6056.38it/s]


## 2.4- Preprocessing Functions

### Here are the functions used to:
- Clean the lyrics, removing stop words and, if desired, using stemming and lemmatization methods available.
- Generate the embedding, divided into chunks to raise accuracy.
- Process the song with a limit of 2000 characters for the sentiment model.

In [6]:
def clean_lyrics(
    text, #lyrics
    language,
    use_stemming=False, #root form of words. i.E: running -> run
    use_lemmatization=True #dictionary form. i.E: better -> good
):
    '''
    This function receives the lyrics from a song as text and the language input,
    If the language is in the language map, the stopwords are removed by a standard list (nltk library).
    
    After the stop words are treated, if the song is in english, we have lemmatization and optionally
    stemming. If the song is not in english, only stemming is available.
    '''

    text = text.lower()

    #Remove texts inside []. i.E: [Intro] 
    text = re.sub(
        r"\[.*?\]",
        "",
        text
    )

    #Removes special characters
    text = re.sub(
        r"[^a-zA-ZÀ-ÿ가-힣\s]",
        "",
        text
    )

    #Stop words languages available.
    language_map = {
        "en": "english",
        "es": "spanish",
        "fr": "french",
        "de": "german",
        "it": "italian",
        "pt": "portuguese"
    }

    if language in language_map:
        #Checks if the language of the song is in the dictionary. If so, uses the stopword function/
        #to remove stopwords.
        try:
            stop_words = set(
                stopwords.words(
                    language_map[language]
                )
            )
        except OSError:
            stop_words = set()
    else:
        stop_words = set()

    tokens = [
        token
        for token in text.split()
        if token not in stop_words
    ]

    #The lemmatization library works mainly for english grammar, so for the other songs/
    #This proccess is skiped.
    if use_lemmatization and language == "en":
        lemmatizer = WordNetLemmatizer()
        tokens = [
            lemmatizer.lemmatize(token)
            for token in tokens
        ]

    #For stemming we have multi-language library
    if use_stemming and language in language_map:
        try:
            stemmer = SnowballStemmer(
                language_map[language]
            )
            tokens = [
                stemmer.stem(token)
                for token in tokens
            ]
        except ValueError:
            pass

    return " ".join(tokens)

In [7]:
def generate_embedding(text, chunk_size=200):
    """
    This function divides the texts of the lyrics into chuncks, to ease the embeding,
    after the embedding, a mean is calculated and used as final value.
    The parameter chunk was decided based on trial and error.

    The embeding uses sentence transformer, initiaded previously.
    """
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(
            words[i:i+chunk_size]
        )
        chunks.append(chunk)

    embeddings = embedding_model.encode(
        chunks
    )

    return np.mean(
        embeddings,
        axis=0
    )

In [8]:
def process_song(filepath):
    """
    Reads the path to the folder, detects the language, runds the embedding,
    return the information of the song.
    """
    with open(
        filepath,
        "r",
        encoding="utf-8"
    ) as f:
        lyrics = f.read()

    try:
        language = detect(lyrics)

    except:
        language = "unknown"

    cleaned_text = clean_lyrics(
        lyrics,
        language
    )

    sentiment = sentiment_model(
        cleaned_text[:2000]
    )[0]

    embedding = generate_embedding(
        cleaned_text
    )

    artist, song = (
        os.path.basename(filepath)
        .replace(".txt","")
        .split(" - ",1)
    )

    return {
        "artist": artist,
        "song": song,
        "language": language,
        "word_count": len(
            cleaned_text.split()
        ),
        "sentiment": sentiment["label"],
        "sentiment_score": sentiment["score"],
        "embedding": embedding
    }

## 2.5- Feature Functions

### In this section are the functions to:
- Run the one hot encoding in the genre collumn.
- Build the matrix used for the recommendations.

In [9]:
def one_hot_encode_columns(
    df,
    columns
):

    encoder = OneHotEncoder(
        sparse_output=False,
        handle_unknown="ignore"
    )

    encoded_matrix = encoder.fit_transform(
        df[columns]
    )

    return encoded_matrix, encoder

In [10]:
def build_recommendation_matrix(df, genre_columns):
    """
    Builds the matrix containing the embedding values, sentiment score and
    genre columns.
    """
    embedding_matrix = np.vstack(
        df["embedding"].values
    )

    sentiment_matrix = df[
        [
            "sentiment_score"
        ]
    ].values

    genre_matrix = df[
        genre_columns
    ].values

    recommendation_matrix = np.hstack(
        [
            embedding_matrix,
            sentiment_matrix,
            genre_matrix
        ]
    )

    return recommendation_matrix

## 2.6- Processing

### In this section we have the function to match the songs in the folder with the songs in the tracks dataset.

### Mainly we loop through all songs, do some slight cleaning to keep unique values only and create a key for the join. This way we can have our dataframe with the lyrics analysis and the genre columns.

### To make sure we were able to match all rows (which should be the case since they are set and subset) we check for cases not matched at the end.

In [11]:
def find_best_track_match(row, score_cutoff=85, track_choices=list):
    """
    Uses the match_key to search inside the songs dataframe for the closes fuzzy match.
    It is used to match the songs in the folder with the songs in the tracks df.
    """
    query = row["match_key"]

    match = process.extractOne(
        query,
        track_choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=score_cutoff
    )

    if match is None:
        return None

    return match[0]

In [14]:
results = []

for file in os.listdir(LYRICS_FOLDER):

    if file.endswith(".txt"):

        result = process_song(
            os.path.join(
                LYRICS_FOLDER,
                file
            )
        )

        results.append(result)

df = pd.DataFrame(results)

tracks_merged = tracks_data.rename(
    columns={"Album Artist":"artist", "Title":"song"}).copy()

df["match_key"] = (
    df["artist"].fillna("") 
    + " - " 
    + df["song"].fillna("")
).str.lower()

tracks_data["match_key"] = (
    tracks_data["Album Artist"].fillna("") 
    + " - " 
    + tracks_data["Title"].fillna("")
).str.lower()

track_choices = tracks_data["match_key"].tolist()

df["matched_key"] = df.apply(
    find_best_track_match,
    axis=1,
    score_cutoff=60,
    track_choices=track_choices
)

tracks_genre = tracks_data[
    [
        "match_key",
        "Genre"
    ]
].drop_duplicates(
    subset=[
        "match_key"
    ]
)

df = df.merge(
    tracks_genre,
    left_on="matched_key",
    right_on="match_key",
    how="left",
    suffixes=("", "_track")
)

missing_genre = df[
    df["Genre"].isna()
][
    [
        "artist",
        "song",
        "matched_key"
    ]
]

display(missing_genre)

#There should not be missing genre, but just in case...
df["Genre"] = df["Genre"].fillna("unknown")

genre_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

genre_encoded = genre_encoder.fit_transform(
    df[
        [
            "Genre"
        ]
    ]
)

genre_columns = genre_encoder.get_feature_names_out(
    [
        "Genre"
    ]
)

genre_df = pd.DataFrame(
    genre_encoded,
    columns=genre_columns,
    index=df.index
)

df = pd.concat(
    [
        df,
        genre_df
    ],
    axis=1
)

display(df.head(10))

display(df.shape)

,artist,song,matched_key
381,Fábio Jr.,Só Você,NaN
626,"Amaarae, Bree Runway & Starkillers",Starkilla,NaN
899,Daniela Mercury,Como Vai Você,NaN
968,BTOB,Wow,NaN
1332,Banda Uó,Cowboy (Vibe Remix),NaN
1728,"Calvin Harris, Justin Timberlake, Halsey & Pha...",Stay With Me,NaN
2061,"Anitta, KING Saints & Rincon Sapiência",Nanã,NaN
2222,Jorge Blanco,Voy por Ti,NaN
2362,Lagum & Cynthia Luz,Eu Não Valho Nada,NaN
2546,"Daft Punk, Pharrell Williams & Nile Rodgers",Get Lucky,NaN


,artist,song,language,word_count,sentiment,sentiment_score,embedding,match_key,matched_key,match_key_track,...,Genre_Samba,Genre_Sertanejo,Genre_Singer/Songwriter,Genre_Soundtrack,Genre_TV Soundtrack,Genre_Teen Pop,Genre_UK Hip-Hop,Genre_Urbano latino,Genre_Worldwide,Genre_unknown
0,Mahmood & Angèle,SEMPRE - JAMAIS,fr,234,neutral,0.401334,"[0.025722805, 0.004459141, 0.032832075, -0.058...",mahmood & angèle - sempre - jamais,mahmood & angèle - sempre / jamais,mahmood & angèle - sempre / jamais,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Mariah Carey,Don't Forget About Us (Radio Edit),en,332,neutral,0.452482,"[-0.037249345, -0.04354989, 0.19697523, -0.041...",mariah carey - don't forget about us (radio edit),mariah carey - don't forget about us (radio edit),mariah carey - don't forget about us (radio edit),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Anitta,Eu Sou do Tipo,pt,118,neutral,0.491574,"[0.14089632, -0.21092568, 0.108326234, 0.03573...",anitta - eu sou do tipo,anitta - eu sou do tipo,anitta - eu sou do tipo,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Bruno Gadiol,"Desculpa, Mas Eu Só Penso Em Você",pt,142,neutral,0.513091,"[0.11875529, 0.18947533, 0.24953532, 0.0461628...","bruno gadiol - desculpa, mas eu só penso em você","bruno gadiol - desculpa, mas eu só penso em você","bruno gadiol - desculpa, mas eu só penso em você",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Lily Allen,Beg For Me,en,179,negative,0.397079,"[0.17473778, -0.20126468, 0.03926919, 0.076895...",lily allen - beg for me,lily allen - beg for me,lily allen - beg for me,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Hugo & Guilherme & Marília Mendonça,Mal Feito,pt,213,neutral,0.466782,"[0.10324251, -0.050316393, 0.17177585, 0.03525...",hugo & guilherme & marília mendonça - mal feito,hugo & guilherme & marília mendonça - mal feito,hugo & guilherme & marília mendonça - mal feito,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,Taylor Swift,All Too Well (Taylor's Version),en,454,neutral,0.423624,"[0.008631964, 0.06300802, 0.19105339, 0.018659...",taylor swift - all too well (taylor's version),taylor swift - all too well (taylor's version),taylor swift - all too well (taylor's version),...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,Paramore,Crushcrushcrush,en,225,neutral,0.428849,"[0.09829045, 0.009286858, 0.16751358, 0.135102...",paramore - crushcrushcrush,paramore - crushcrushcrush,paramore - crushcrushcrush,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,Tove Lo,I’m to Blame,en,147,neutral,0.422778,"[-0.034748904, 0.14821926, 0.2784112, 0.235718...",tove lo - i’m to blame,tove lo - i’m to blame,tove lo - i’m to blame,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,Anitta,Aceita,es,135,neutral,0.453560,"[-0.14637595, -0.189817, 0.19458532, -0.139486...",anitta - aceita,anitta - aceita,anitta - aceita,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(2722, 60)

### 2.7- Saving a quick run backup for testing and debugging.

### In this section we save the dataframe created previously so we can visualize and use for debugging if needed.

In [15]:
#Pickle is a type of python file that we use here for efficiency.
df.to_pickle(
    "lyrics_dataset.pkl"
)

## 2.8- Recommendation Functions

### In this section we have our recommendation functions and the function to embed our query for semantic search.

### We also have our semantic search function, so we can do recommendations based on a description given by the user.

In [16]:
def recommend_song(
    song_name,
    top_n=10
):

    recommendation_matrix = build_recommendation_matrix(
        df,
        genre_columns
    )

    idx = df[
        df["song"] == song_name
    ].index[0]

    target_features = recommendation_matrix[idx]

    #Here we work with the cosine_similarity since the direction of the similarity is important for us.
    similarities = cosine_similarity(
        [
            target_features
        ],
        recommendation_matrix
    )[0]

    temp_df = df.copy()

    temp_df["similarity"] = similarities

    recommendations = (
        temp_df
        .sort_values(
            "similarity",
            ascending=False
        )
        .iloc[1:top_n+1]
    )

    return recommendations[
        [
            "artist",
            "song",
            "Genre",
            "sentiment",
            "similarity"
        ]
    ]

In [17]:
def embed_query(query):

    embedding = embedding_model.encode(
        query,
        convert_to_numpy=True
    )

    return embedding

In [18]:
def semantic_search(
    query,
    genre=None,
    top_n=10
):

    query_embedding = embed_query(query)

    #Limit of 2000 characters in the search.
    sentiment = sentiment_model(
        query[:2000]
    )[0]

    #Transform the query into a score.
    query_sentiment_score = np.array(
        [
            [
                sentiment["score"]
            ]
        ]
    )

    #If the user provides a genre, use it for the search.
    if genre is None: #Creates a row of 0s
        query_genre_matrix = np.zeros(
            (
                1,
                len(genre_columns)
            )
        )
    else: #Add the genre to the dataframe
        query_genre_df = pd.DataFrame(
            {
                "Genre": [
                    genre
                ]
            }
        )
        #One hot encode the genre vector.
        query_genre_matrix = genre_encoder.transform(
            query_genre_df[
                [
                    "Genre"
                ]
            ]
        )

    query_features = np.hstack(
        [
            query_embedding.reshape(1, -1), #makes the embeding one single row i.E: [[0.12, -0.04, 0.88, ...]]
            query_sentiment_score,
            query_genre_matrix
        ]
    )

    recommendation_matrix = build_recommendation_matrix(
        df,
        genre_columns
    )

    similarities = cosine_similarity(
        query_features,
        recommendation_matrix
    )[0]

    results = df.copy()

    results["similarity"] = similarities

    results = (
        results
        .sort_values(
            "similarity",
            ascending=False
        )
        .head(top_n)
    )

    return results[
        [
            "artist",
            "song",
            "Genre",
            "language",
            "sentiment",
            "similarity"
        ]
    ]

## 2.9- Demo and Results

### In this section we run some tests to see how our system works in practice.

### Recommendation based on another song:

- Will recommend songs based on another one that is already in our Lyrics Folder. Please note the name must be exact as in the folder.

In [21]:
recommend_song(
    "...Baby One More Time",
    top_n=20
)

,artist,song,Genre,sentiment,similarity
219,Ashley Tisdale,What If,Pop,neutral,0.834673
1740,Leah Kate,Cry Baby,Pop,neutral,0.823280
1969,Christian Leave,Please Notice,Pop,neutral,0.822757
2132,Leyla Blue,What A Shame,Pop,negative,0.818762
881,Leah Kate,Boy Next Door,Pop,neutral,0.807773
302,Zara Larsson,Stick With You,Pop,neutral,0.805178
1558,Beyoncé,Love On Top,Pop,neutral,0.801364
85,DUDA BEAT,BABY,Pop,neutral,0.796937
1828,Ariana Grande,Into You,Pop,neutral,0.795523
2045,Beyoncé,Why Don't You Love Me,Pop,neutral,0.793548


In [25]:
recommend_song(
    "So It Goes...",
    top_n=20
)

,artist,song,Genre,sentiment,similarity
1057,Taylor Swift,Style (Taylor's Version),Pop,neutral,0.750377
1393,Taylor Swift,Style,Pop,neutral,0.750377
2180,Olivia Rodrigo,get him back!,Pop,neutral,0.738182
1930,Zara Larsson,Look What You've Done,Pop,neutral,0.722630
878,Pabllo Vittar,Triste com T,Pop,neutral,0.720787
2453,Pabllo Vittar & Gloria Groove,AMEIANOITE,Pop,neutral,0.719643
1147,Austin Mahone & Pitbull,Mmm Yeah,Pop,neutral,0.711239
231,DENNIS,Bate,Pop,neutral,0.708984
1563,Miley Cyrus,Night Crawling (feat. Billy Idol),Pop,neutral,0.706911
896,PinkPantheress,Girl Like Me,Pop,neutral,0.700897


### Semantic Search Recommendation:
- Recommend based on a phrase describing the emotion expected or the idea expected from the song.
- Good example of phrase for search is: "songs about heartbreak and still missing someone"
- Can also use the gender to find similars

In [27]:
semantic_search(
    "songs about heartbreak and still missing someone",
    genre="Alternative"
)

,artist,song,Genre,language,sentiment,similarity
1863,Taylor Swift,A Message From Taylor,Pop,en,neutral,0.621388
1774,Taylor Swift,hoax,Alternative,en,neutral,0.612168
1651,Gal Gosta,O Amor Em Paz,Latin Jazz,pt,negative,0.566702
616,Marina Sena,SENSEI,Pop,pt,neutral,0.558774
545,Adele,Someone Like You,Pop,en,neutral,0.558283
513,Flávio Venturini,Pra lembrar de nós,Brazilian,pt,neutral,0.556248
727,half•alive,still feel.,Alternative,en,neutral,0.551145
1192,Marisa Monte,Ainda Bem,MPB,pt,neutral,0.544718
1428,Madonna,Fragile,Pop,en,negative,0.544076
1135,Taylor Swift,marjorie,Alternative,en,neutral,0.543291


In [28]:
semantic_search(
    "songs about heartbreak and still missing someone",
    genre="Pop"
)

,artist,song,Genre,language,sentiment,similarity
1863,Taylor Swift,A Message From Taylor,Pop,en,neutral,0.673569
616,Marina Sena,SENSEI,Pop,pt,neutral,0.618644
545,Adele,Someone Like You,Pop,en,neutral,0.606304
2297,Anavitória,"Ai, Amor",Pop,pt,neutral,0.595686
1428,Madonna,Fragile,Pop,en,negative,0.594248
1381,Lagum,NINGUÉM ME ENSINOU,Pop,pt,neutral,0.585917
721,TIAGO IORC,Sol Que Faltava,Pop,pt,negative,0.577329
1583,Jão,Ainda Te Amo,Pop,pt,neutral,0.570133
512,Mariah Carey,Hero (2009 Version),Pop,en,neutral,0.568774
1651,Gal Gosta,O Amor Em Paz,Latin Jazz,pt,negative,0.566702


In [29]:
semantic_search(
    "energetic dance song about desire and attraction",
    genre="Dance"
)

,artist,song,Genre,language,sentiment,similarity
1890,MC Leozinho & DJ Marlboro,Ela Só Pensa Em Beijar,Funk,pt,neutral,0.586653
1297,Rainbow,Let's Dance,K-Pop,en,positive,0.557363
356,HyunA,어때,K-Pop,ko,neutral,0.540468
2650,Caetano Veloso,Samba de Verão,MPB,pt,neutral,0.527712
542,Vanessa da Mata,Amado,Latin,pt,neutral,0.526438
1429,Taylor Swift,Holy Ground (Taylor's Version),Pop,en,neutral,0.521843
2262,Pabllo Vittar,Corpo Sensual (feat. Mateus Carrilho),MPB,pt,neutral,0.518895
1686,"Calvin Harris, Dua Lipa & Young Thug",Potion,Dance,en,neutral,0.516920
717,Anitta,Cravo e Canela (feat. Vitin),Pop,pt,neutral,0.514300
1062,Tove Lo & SG Lewis,Desire,Dance,en,neutral,0.512753


In [30]:
semantic_search(
    "energetic dance song about desire and attraction",
    genre="Pop"
)

,artist,song,Genre,language,sentiment,similarity
1890,MC Leozinho & DJ Marlboro,Ela Só Pensa Em Beijar,Funk,pt,neutral,0.586653
717,Anitta,Cravo e Canela (feat. Vitin),Pop,pt,neutral,0.578021
1429,Taylor Swift,Holy Ground (Taylor's Version),Pop,en,neutral,0.573260
1297,Rainbow,Let's Dance,K-Pop,en,positive,0.557363
2297,Anavitória,"Ai, Amor",Pop,pt,neutral,0.556722
1263,Taylor Swift,Suburban Legends (Taylor's Version) [From The ...,Pop,en,neutral,0.547960
49,Anitta,Gosto Assim (feat. Dubeat),Pop,pt,neutral,0.545341
630,Agnetha Fältskog,Dance Your Pain Away,Pop,en,neutral,0.543352
1382,"Pabllo Vittar, MC Tchelinho & Frimes",Apetitosa (S4TAN Remix),Pop,pt,neutral,0.542265
356,HyunA,어때,K-Pop,ko,neutral,0.540468


# 3- Conclusion

## Evaluation Idea

### The evaluation method designed for this project is the following: 
- After a recommendation is generated for a user, we ask for their response, either a like or a dislike.
    - The songs recommendeded will get the vote from the user based on what it was associated with, for example:
    If an user asks for melancolis songs and like the playlist, all 10 songs will get a like and be registered as melancolic score. 

    - Then if more people asks for melancolic and get some of the previous 10 songs, a mean will be calculated if they like it or not.

    - The means will be used to tune our system in regard of the genre and semantic search.

### The system successfully creates playlists based on the lyrics themes and genre. The searching method also works with no problem.

### A topic to be considered is how the data ingection could be updated in order to have a more dinamic system.

# 4- Reference

### To extract the lyrics from the songs, I had as a baseline the repository: https://github.com/b5i/apple-playlist-lyrics-downloader